# Stage 4: summary and verdict

Everything from Stages 1 to 3 and the cross-validation in one place. **No training here**: this notebook only reads saved results.

1. The results table: 6 models x 2 tasks x 3 scores, from cross-validation (151 people) and from the single test (30 people)
2. Which differences are real? Person-by-person comparisons with 95% ranges
3. What matters most for SilverGuard: still vs moving, the oldest people, and the rare active class
4. Model sizes
5. The verdict, and what it does *not* tell us yet

**Kernel:** `.venv-tf` in this folder. Run all earlier notebooks first.

## How to read this notebook (for beginners)

**Two kinds of evidence:**

| | Single test (Stages 1–3) | Cross-validation |
|---|---|---|
| Tested on | 30 hidden people | all 151 people, each tested once by a model that never saw them |
| How trustworthy | OK, but 30 people leaves room for luck | **better**: 5x more people, and we can see how much the score moves from round to round |

So **cross-validation is the main evidence**, and the single test is a second opinion. If both say the same thing, we can be fairly confident.

**The six models:** three ways of reading the wrist signal (random forest, CNN, XGBoost), each with and without the **HMM**, the smoothing step that stops the label from flipping every 30 seconds.

**Kappa:** 0 = no better than guessing, 1 = perfect. It removes the accuracy you'd get by luck, so a model can't score well by always guessing the most common activity.

**"Tie" vs "better":** for two models, each person gets a score from both. We average the differences and find a 95% range by resampling people (bootstrap). If the range is entirely above 0, model A is genuinely better. If it includes 0, the gap could be luck, so we call it a tie.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb

from common import BINARY, RESULTS_DIR, confusion_table, paired_bootstrap, scores, to_binary

MODELS = ['RF', 'RF+HMM', 'CNN', 'CNN+HMM', 'XGB', 'XGB+HMM']
SCORES = ['accuracy', 'macro-F1', 'kappa']

cv = pd.read_csv(RESULTS_DIR / 'cv_metrics.csv')
single = pd.concat([pd.read_csv(RESULTS_DIR / f) for f in ['stage1_metrics.csv', 'stage2_metrics.csv', 'stage3_xgb_metrics.csv']])
cv_people = pd.read_csv(RESULTS_DIR / 'cv_per_person.csv')
pairwise_cv = pd.read_csv(RESULTS_DIR / 'cv_pairwise.csv')
trees = np.load(RESULTS_DIR / 'cv_trees_predictions.npz')
cnn = np.load(RESULTS_DIR / 'cv_cnn_predictions.npz')
assert (trees['pid'] == cnn['pid']).all() and (trees['y'] == cnn['y']).all()
y = trees['y']
pred = {n: trees[f'pred_{n}'] for n in ['RF', 'RF+HMM', 'XGB', 'XGB+HMM']}
pred.update({n: cnn[f'pred_{n}'] for n in ['CNN', 'CNN+HMM']})
assert len(cv_people) == 151 and set(cv.model) == set(MODELS)
print('loaded results for', len(cv_people), 'people')

loaded results for 151 people


## 1. The results table

**Baseline:** "always sedentary" answers the most common 4-class activity every time. It shows what doing nothing clever scores.

In the cross-validation, "RF" is XGBoost's random-forest mode on the GPU (the no-CPU-training rule). On the single test it scored almost the same as the Stage 1 forest (4-class kappa 0.693 vs 0.701; see `cv_trees.ipynb`).

In [2]:
# the baseline for the cross-validation: always answer 'sedentary' (class 1)
always = np.ones_like(y)
baseline = [{'model': 'always sedentary', 'task': '4-class', **scores(y, always, 4)},
            {'model': 'always sedentary', 'task': 'binary', **scores(to_binary(y), to_binary(always), 2)}]
cv_all = pd.concat([pd.DataFrame(baseline), cv])
order = ['always sedentary'] + MODELS

table = pd.concat({'cross-validation (151 people)': cv_all.set_index(['model', 'task'])[SCORES],
                   'single test (30 people)': single.set_index(['model', 'task'])[SCORES]}, axis=1)
table.to_csv(RESULTS_DIR / 'stage4_summary_table.csv')
for task in ['4-class', 'binary']:
    print(f'===== {task} =====')
    print(table.xs(task, level='task').loc[order].round(3).to_string(), '\n')

===== 4-class =====
                 cross-validation (151 people)                 single test (30 people)                
                                      accuracy macro-F1  kappa                accuracy macro-F1  kappa
model                                                                                                 
always sedentary                         0.396    0.142  0.000                   0.414    0.146  0.000
RF                                       0.803    0.749  0.707                   0.806    0.741  0.701
RF+HMM                                   0.883    0.822  0.825                   0.880    0.801  0.814
CNN                                      0.788    0.718  0.683                   0.776    0.710  0.659
CNN+HMM                                  0.859    0.781  0.788                   0.848    0.784  0.768
XGB                                      0.802    0.744  0.707                   0.800    0.741  0.696
XGB+HMM                                  0.881    0.8

## 2. Which differences are real?

Person-by-person kappa differences, with a 95% range. First the cross-validation (151 people), then the same comparisons on the single test (30 people).

In [3]:
def compare(people, pairs, n_people):
    rows = []
    for a, b in pairs:
        for t in ['4-class', 'binary']:
            both = people[[f'{a} {t}', f'{b} {t}']].dropna()
            mean, lo, hi = paired_bootstrap(both.iloc[:, 0], both.iloc[:, 1])
            rows.append({'A': a, 'B': b, 'task': t, f'A better for (of {n_people})': int((both.iloc[:, 0] > both.iloc[:, 1]).sum()),
                         'mean kappa A - B': round(mean, 4), '95% range': f'{lo:+.3f} to {hi:+.3f}',
                         'verdict': 'A better' if lo > 0 else ('B better' if hi < 0 else 'tie')})
    return pd.DataFrame(rows)

MAIN_PAIRS = [('XGB+HMM', 'RF+HMM'), ('RF+HMM', 'CNN+HMM'), ('XGB+HMM', 'CNN+HMM')]
cv_compare = compare(cv_people, MAIN_PAIRS, 151)
print('cross-validation, 151 people:')
print(cv_compare.to_string(index=False))

# the single test: Stage 2 saved RF and CNN per person, Stage 3 saved the HMM models and XGB
s2 = pd.read_csv(RESULTS_DIR / 'stage2_per_person.csv').set_index('pid')
s3 = pd.read_csv(RESULTS_DIR / 'stage3_xgb_per_person.csv').set_index('pid')
single_people = s3.join(s2[['RF 4-class', 'RF binary', 'CNN 4-class', 'CNN binary']])
assert len(single_people) == 30
single_compare = compare(single_people, MAIN_PAIRS, 30)
print('\nsingle test, 30 people:')
print(single_compare.to_string(index=False))

pd.concat({'cross-validation': cv_compare, 'single test': single_compare}).to_csv(RESULTS_DIR / 'stage4_comparisons.csv')

cross-validation, 151 people:
      A       B    task  A better for (of 151)  mean kappa A - B        95% range  verdict
XGB+HMM  RF+HMM 4-class                     80           -0.0039 -0.010 to +0.002      tie
XGB+HMM  RF+HMM  binary                     94            0.0190 +0.010 to +0.030 A better
 RF+HMM CNN+HMM 4-class                    105            0.0392 +0.028 to +0.051 A better
 RF+HMM CNN+HMM  binary                     79           -0.0014 -0.015 to +0.012      tie
XGB+HMM CNN+HMM 4-class                    100            0.0353 +0.023 to +0.048 A better
XGB+HMM CNN+HMM  binary                     92            0.0176 +0.003 to +0.033 A better



single test, 30 people:
      A       B    task  A better for (of 30)  mean kappa A - B        95% range  verdict
XGB+HMM  RF+HMM 4-class                    19            0.0024 -0.012 to +0.017      tie
XGB+HMM  RF+HMM  binary                    16            0.0228 -0.002 to +0.055      tie
 RF+HMM CNN+HMM 4-class                    22            0.0501 +0.020 to +0.083 A better
 RF+HMM CNN+HMM  binary                    18            0.0130 -0.025 to +0.051      tie
XGB+HMM CNN+HMM 4-class                    25            0.0526 +0.027 to +0.082 A better
XGB+HMM CNN+HMM  binary                    18            0.0358 +0.003 to +0.072 A better


**What the HMM adds** (cross-validation): for each model, how many of the 151 people improve when the HMM smooths the day.

In [4]:
print(pairwise_cv[pairwise_cv.B.isin(['RF', 'CNN', 'XGB'])][['A', 'B', 'task', 'A better for', 'mean kappa A - B', '95% low', '95% high']]
      .round(3).to_string(index=False))

      A   B    task  A better for  mean kappa A - B  95% low  95% high
 RF+HMM  RF 4-class           151             0.123    0.115     0.130
 RF+HMM  RF  binary           138             0.078    0.068     0.087
CNN+HMM CNN 4-class           151             0.107    0.099     0.116
CNN+HMM CNN  binary           138             0.051    0.043     0.059
XGB+HMM XGB 4-class           150             0.120    0.112     0.127
XGB+HMM XGB  binary           146             0.090    0.081     0.101


## 3. What matters most for SilverGuard

**a) Still vs moving.** The watch's main question. Two mistakes matter:

- **False alarm:** the person was still but the model says "moving"
- **Missed movement:** the person was moving but the model says "still"

In [5]:
rows = []
for n in ['RF+HMM', 'CNN+HMM', 'XGB+HMM']:
    t, p = to_binary(y), to_binary(pred[n])
    rows.append({'model': n, 'false alarms (% of still)': 100 * ((p == 1) & (t == 0)).sum() / (t == 0).sum(),
                 'movement found (% of moving)': 100 * ((p == 1) & (t == 1)).sum() / (t == 1).sum(),
                 'binary kappa': scores(t, p, 2)['kappa']})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print('\nXGB+HMM, still vs moving, all 151 people:')
print(confusion_table(to_binary(y), to_binary(pred['XGB+HMM']), BINARY).to_string())

  model  false alarms (% of still)  movement found (% of moving)  binary kappa
 RF+HMM                      4.764                        84.971         0.799
CNN+HMM                      4.407                        84.406         0.802
XGB+HMM                      5.631                        89.206         0.812

XGB+HMM, still vs moving, all 151 people:


              still  moving  still %  moving %
true still   219543   13100     94.4       5.6
true moving    7396   61125     10.8      89.2

**b) The oldest people.** SilverGuard is for elderly users. Capture-24's oldest group is 53+ (34 of 151 people), still younger than most SilverGuard users, but the closest we have.

In [6]:
old = cv_people[cv_people.age == '53+']
print(f'median kappa per person, 53+ (n={len(old)}) vs under 53 (n={len(cv_people) - len(old)}):')
print(pd.DataFrame({f'{g} {t}': {n: d[f'{n} {t}'].median() for n in ['RF+HMM', 'CNN+HMM', 'XGB+HMM']}
                    for g, d in [('53+', old), ('under 53', cv_people[cv_people.age != '53+'])] for t in ['4-class', 'binary']})
      .round(3).to_string())
old_compare = compare(old, MAIN_PAIRS, len(old))
old_compare.to_csv(RESULTS_DIR / 'stage4_comparisons_53plus.csv', index=False)
print('\n53+ only:')
print(old_compare.to_string(index=False))

median kappa per person, 53+ (n=34) vs under 53 (n=117):
         53+ 4-class  53+ binary  under 53 4-class  under 53 binary
RF+HMM         0.820       0.800             0.840            0.807
CNN+HMM        0.737       0.808             0.799            0.796
XGB+HMM        0.831       0.853             0.834            0.827

53+ only:
      A       B    task  A better for (of 34)  mean kappa A - B        95% range  verdict
XGB+HMM  RF+HMM 4-class                    23            0.0052 -0.011 to +0.022      tie
XGB+HMM  RF+HMM  binary                    26            0.0356 +0.016 to +0.062 A better
 RF+HMM CNN+HMM 4-class                    26            0.0596 +0.034 to +0.089 A better
 RF+HMM CNN+HMM  binary                    17           -0.0227 -0.059 to +0.009      tie
XGB+HMM CNN+HMM 4-class                    27            0.0648 +0.038 to +0.094 A better
XGB+HMM CNN+HMM  binary                    20            0.0129 -0.018 to +0.042      tie


**c) The rare active class.** Moderate-vigorous is about 5% of the data, but it's what "was this person active today?" depends on.

In [7]:
from sklearn.metrics import f1_score, recall_score
CLASSES = ['sleep', 'sedentary', 'light', 'moderate-vigorous']
per_class = pd.DataFrame({n: f1_score(y, pred[n], labels=range(4), average=None) for n in MODELS}, index=CLASSES).T
per_class['moderate-vigorous found (recall)'] = [recall_score(y, pred[n], labels=[3], average=None)[0] for n in MODELS]
print('F1 per class (cross-validation), and how much moderate-vigorous each finds:')
print(per_class.loc[MODELS].round(3).to_string())

F1 per class (cross-validation), and how much moderate-vigorous each finds:
         sleep  sedentary  light  moderate-vigorous  moderate-vigorous found (recall)
RF       0.904      0.798  0.666              0.629                             0.656
RF+HMM   0.971      0.886  0.743              0.688                             0.638
CNN      0.884      0.777  0.666              0.544                             0.559
CNN+HMM  0.947      0.863  0.727              0.588                             0.537
XGB      0.904      0.796  0.671              0.603                             0.638
XGB+HMM  0.964      0.883  0.761              0.675                             0.614


## 4. Model sizes

The file on the laptop, not yet the size on the watch (that comes with export, later). The HMM adds almost nothing: 4 start numbers and a 4 x 4 table.

In [8]:
booster = xgb.Booster()
booster.load_model(RESULTS_DIR / 'xgb_4class.json')
xgb_nodes = sum(len(t.splitlines()) for t in booster.get_dump())
sizes = pd.DataFrame([
    {'model': 'RF (Stage 1, scikit-learn)', 'file': 'rf_4class.joblib', 'what is inside': '200 deep trees'},
    {'model': 'CNN', 'file': 'cnn_4class.keras', 'what is inside': '36,580 numbers'},
    {'model': 'XGB', 'file': 'xgb_4class.json', 'what is inside': f'800 trees of depth up to 8, {xgb_nodes:,} nodes in total'},
    {'model': 'HMM', 'file': 'hmm.npz', 'what is inside': '4 + 16 numbers'}])
sizes['MB'] = [round((RESULTS_DIR / f).stat().st_size / 1e6, 2) for f in sizes.file]
print(sizes.to_string(index=False))

                     model             file                                     what is inside     MB
RF (Stage 1, scikit-learn) rf_4class.joblib                                     200 deep trees 136.15
                       CNN cnn_4class.keras                                     36,580 numbers   0.50
                       XGB  xgb_4class.json 800 trees of depth up to 8, 237,416 nodes in total  14.92
                       HMM          hmm.npz                                     4 + 16 numbers   0.00


## 5. Verdict

**1. The HMM matters more than the choice of model.** Smoothing the day adds +0.107 (CNN) to +0.123 (forest) kappa on 4 classes, and helps 150–151 of the 151 people for every model. Whatever model SilverGuard uses, it should be followed by the HMM (or something like it).

**2. Best results: XGBoost + HMM.**
- 4-class kappa **0.822** (cross-validation): a **tie** with the random forest + HMM (0.825)
- Still vs moving kappa **0.812**: **better** than the forest (+0.019, 95% range entirely above 0). It finds more of the movement (89% vs 85%) at the cost of a few more false alarms (5.6% vs 4.8% of still windows called moving)
- Better than the CNN + HMM on both tasks
- 53+ group (34 people): ties the forest on 4 classes, **better** on still vs moving (26 of 34 people, +0.036)
- Moderate-vigorous: slightly behind the forest (F1 0.675 vs 0.688). On the single test it was ahead, so this one is close and not settled
- 15 MB, about 9x smaller than the scikit-learn forest

**3. Random forest + HMM: equally accurate on 4 classes, but 136 MB.** It is the only model with no clear advantage left: XGBoost matches it on 4 classes, beats it on still vs moving, and is much smaller.

**4. The 1D CNN + HMM did not beat the tree models.** That was the main question of this project.
- 4 classes: **worse** than both tree models (about −0.035 to −0.039 kappa, 95% ranges below 0)
- Still vs moving: **tied** with the forest, behind XGBoost (−0.018, 95% range just below 0)
- **Clearly weaker for the 53+ group on 4 classes** (median kappa 0.737 vs 0.820–0.831), though tied with both on still vs moving for them
- **Weakest on moderate-vigorous activity** (F1 0.588 vs 0.675–0.688, and it finds only 54% of it)
- Its big advantage is size: **0.5 MB**, 30x smaller than XGBoost

**5. Both kinds of evidence agree.** The single 30-person test and the 151-person cross-validation rank the models the same way, and the cross-validation scores are 0.004–0.046 higher than the single test. With 5x more people, the cross-validation can also settle what the single test could not: XGBoost + HMM's lead on still vs moving was a tie on 30 people, and is real on 151.

### Recommendation for SilverGuard
**XGBoost + HMM is the best candidate**, if it fits on the watch. The current model is 15 MB as a text file. A smaller version (depth 4 scored almost the same in Stage 3's tuning) and a compact export should shrink it a lot; that has to be checked at export. **The CNN is the fallback** if size forces it, but it would need work first, mainly for older people and active minutes, before it can be trusted for elderly users.

### What this does *not* tell us yet
- **The HMM hides short bursts of activity.** It lowers how much moderate-vigorous activity is found for every model (forest 66% → 64%, XGBoost 64% → 61%, CNN 56% → 54%), because a short burst looks like a mistake to smooth away. Older people's active bouts are often short, so this needs a closer look
- **The HMM uses the whole day**, including what comes *after* each window. On the watch, live, it can only use the past. A live (forward-only) version needs testing
- **Only labelled windows are scored.** Unlabelled stretches and windows of mixed activity were left out
- **Capture-24's oldest group is 53+**, and most SilverGuard users will be older. Testing on older people (for example GOTOV) is still to do
- **Capture-24 is not the SilverGuard watch.** Different sensor (Axivity vs MPU-6050), same wrist but different hardware
- **One CNN design, one seed.** A better CNN might close the gap; we tested a fair, simple one, not the best possible one
- **Settings were chosen on 121 of the 151 people** before the cross-validation, and XGBoost was added after the single test had been seen once. Both are small advantages, noted for honesty
- **In the cross-validation, "RF" is the GPU forest**, a close stand-in for the Stage 1 forest, not the same model

### Checking the verdict against the numbers

Every claim above is checked here. If the results ever change and a claim stops being true, this cell fails.

In [9]:
v = cv_compare.set_index(['A', 'B', 'task'])['verdict']
k = cv.set_index(['model', 'task'])['kappa']

# 1. the HMM helps every model, for nearly everyone
hmm_gain = pairwise_cv[pairwise_cv.B.isin(['RF', 'CNN', 'XGB']) & (pairwise_cv.task == '4-class')]
assert hmm_gain['mean kappa A - B'].between(0.105, 0.125).all() and (hmm_gain['95% low'] > 0.09).all()
assert (hmm_gain['A better for'] >= 150).all()
# 2. XGB+HMM: ties RF+HMM on 4-class, beats it on binary, beats CNN+HMM on both
assert v['XGB+HMM', 'RF+HMM', '4-class'] == 'tie' and v['XGB+HMM', 'RF+HMM', 'binary'] == 'A better'
assert v['XGB+HMM', 'CNN+HMM', '4-class'] == 'A better' and v['XGB+HMM', 'CNN+HMM', 'binary'] == 'A better'
assert round(k['XGB+HMM', '4-class'], 3) == 0.822 and round(k['XGB+HMM', 'binary'], 3) == 0.812
# 4. CNN+HMM: worse than RF+HMM on 4-class, tied on binary; weaker for 53+
assert v['RF+HMM', 'CNN+HMM', '4-class'] == 'A better' and v['RF+HMM', 'CNN+HMM', 'binary'] == 'tie'
assert old['CNN+HMM 4-class'].median() < old['XGB+HMM 4-class'].median() - 0.05
assert per_class.loc['CNN+HMM', 'moderate-vigorous'] < per_class.loc['XGB+HMM', 'moderate-vigorous'] < per_class.loc['RF+HMM', 'moderate-vigorous']
o = old_compare.set_index(['A', 'B', 'task'])['verdict']
assert o['XGB+HMM', 'RF+HMM', '4-class'] == 'tie' and o['XGB+HMM', 'RF+HMM', 'binary'] == 'A better'
assert o['RF+HMM', 'CNN+HMM', 'binary'] == 'tie' and o['XGB+HMM', 'CNN+HMM', 'binary'] == 'tie'
# the HMM lowers moderate-vigorous recall for every model
r = per_class['moderate-vigorous found (recall)']
assert all(r[f'{m}+HMM'] < r[m] for m in ['RF', 'CNN', 'XGB'])
# XGB+HMM vs RF+HMM on binary: a tie on the single test, real on cross-validation
sv = single_compare.set_index(['A', 'B', 'task'])['verdict']
assert sv['XGB+HMM', 'RF+HMM', 'binary'] == 'tie'
# 5. the single test ranks the HMM models the same way on 4-class
s = single.set_index(['model', 'task'])['kappa']
assert s['CNN+HMM', '4-class'] < min(s['RF+HMM', '4-class'], s['XGB+HMM', '4-class'])
assert s['XGB+HMM', 'binary'] > s['RF+HMM', 'binary'] > s['CNN+HMM', 'binary']
gap = (table[('cross-validation (151 people)', 'kappa')] - table[('single test (30 people)', 'kappa')]).loc[MODELS]
assert gap.between(0, 0.05).all(), gap
print('every claim in the verdict matches the numbers')

every claim in the verdict matches the numbers


## 6. Save

In [10]:
import nbformat
verdict_text = next(x.source for x in nbformat.read('stage4_summary.ipynb', 4).cells if x.metadata.get('verdict'))
(RESULTS_DIR / 'stage4_verdict.md').write_text(verdict_text, encoding='utf-8')
for f in sorted(RESULTS_DIR.glob('stage4_*')):
    print(f'{f.name:34s} {f.stat().st_size / 1e3:8.1f} kB')

stage4_comparisons.csv                  1.0 kB
stage4_comparisons_53plus.csv           0.4 kB
stage4_summary_table.csv                2.0 kB
stage4_verdict.md                       4.0 kB
